# 🎧 DJ Set → Clips 9:16 (mode DJ)

Découpe automatiquement un DJ set en clips verticaux pour TikTok, Reels et Shorts.
Les moments sont choisis **à l'oreille du set** : retour des basses après un breakdown (drops), puis pics d'énergie.
Aucune clé API nécessaire.

**Structure de chaque clip :** hook de 3 s sur le drop → transition → 8 mesures de montée → le drop → 16 mesures pleine énergie.

**Mode d'emploi :** menu *Exécution › Modifier le type d'exécution* → **GPU T4** (plus rapide, pas obligatoire), puis exécute les cellules dans l'ordre.

Si tu relances après une erreur d'installation : menu *Exécution › Déconnecter et supprimer l'environnement d'exécution*, puis *Tout exécuter*.

In [ ]:
#@title 1. Installation (~1 min)
%cd /content
!rm -rf /content/opensource-clipping
!git clone -q https://github.com/djaykoi/opensource-clipping.git /content/opensource-clipping
%cd /content/opensource-clipping
!pip install -q -r requirements-dj.txt
print("✅ Installation terminée")

## 2. Réglages

- **Source YouTube** : colle le lien du set. Si YouTube bloque le téléchargement (erreur 403 / « sign in to confirm »), utilise la source Drive.
- **Source Drive** : dépose le fichier vidéo du set dans ton Google Drive et indique son chemin, par ex. `/content/drive/MyDrive/HHWM 9/Horizontal.mp4`.

In [ ]:
#@title 2. Réglages
SOURCE = "youtube"  #@param ["youtube", "drive"]
LIEN_YOUTUBE = "https://youtu.be/6OG4cVp23Qk"  #@param {type:"string"}
CHEMIN_DRIVE = "/content/drive/MyDrive/HHWM 9/Horizontal.mp4"  #@param {type:"string"}

TITRE = "Hello World Mix 9"  #@param {type:"string"}
HASHTAGS = "#afrohouse #djset #helloworldmix"  #@param {type:"string"}
NB_CLIPS = 9  #@param {type:"slider", min:1, max:20, step:1}
# Premier titre du set : OBLIGATOIRE, toujours le clip n°1 (début-fin du titre)
PREMIER_TITRE = "0:00-2:50"  #@param {type:"string"}
NOM_PREMIER_TITRE = "Amina - D Jay Koi"  #@param {type:"string"}
TEXTE_MINIATURE = "HELLO WORLD MIX"  #@param {type:"string"}
TRANSITION = "mire"  #@param ["mire", "glitch"]
FORMAT = "9:16"  #@param ["9:16", "4:5", "1:1"]
MESURES_AVANT_DROP = 8  #@param {type:"integer"}
MESURES_APRES_DROP = 16  #@param {type:"integer"}
ECART_MIN_SECONDES = 90  #@param {type:"integer"}
HOOK_SECONDES = 3  #@param {type:"integer"}
BPM_FORCE = 0  #@param {type:"number"}
LOGO_WATERMARK = ""  #@param {type:"string"}

assert PREMIER_TITRE.strip(), "⛔ PREMIER_TITRE est obligatoire (ex. 0:00-6:00) : le premier titre du set est toujours le clip n°1."

if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
print("✅ Réglages enregistrés")

In [ ]:
#@title 3. Lancer le découpage
import shlex, os
src_args = (["--url", CHEMIN_DRIVE, "--source", "local"] if SOURCE == "drive"
            else ["--url", LIEN_YOUTUBE, "--source", "youtube"])
args = src_args + [
    "--dj-mode",
    "--clips", str(NB_CLIPS),
    "--ratio", FORMAT,
    "--hook-duration", str(HOOK_SECONDES),
    "--dj-title", TITRE,
    "--dj-hashtags", HASHTAGS,
    "--dj-pre-bars", str(MESURES_AVANT_DROP),
    "--dj-post-bars", str(MESURES_APRES_DROP),
    "--dj-min-gap", str(ECART_MIN_SECONDES),
    "--source-height", "1080",
    "--transition", TRANSITION,
    "--dj-thumb-text", TEXTE_MINIATURE,
]
assert PREMIER_TITRE.strip(), "⛔ PREMIER_TITRE est obligatoire : relance la cellule 2."
args += ["--dj-include", PREMIER_TITRE.split(",")[0].strip(), "--dj-include-label", NOM_PREMIER_TITRE or "Intro"]
if BPM_FORCE:
    args += ["--dj-bpm", str(BPM_FORCE)]
if LOGO_WATERMARK and os.path.exists(LOGO_WATERMARK):
    args += ["--watermark", "--image", LOGO_WATERMARK, "--position", "top-right", "--opacity", "80"]
cmd = "python main.py " + " ".join(shlex.quote(a) for a in args)
print(cmd)
!{cmd}

In [ ]:
#@title 4. Résultat : courbe d'énergie et aperçu des clips
from IPython.display import Image, Video, display
import glob, json
display(Image("outputs/dj_energy.png"))
for c in sorted(json.load(open("outputs/gemini_response.json")), key=lambda c: c["rank"]):
    print(f"#{c['rank']} {c['title_inggris']} | {c['alasan']}")
for f in sorted(glob.glob("outputs/highlight_rank_*_ready.mp4"))[:3]:
    print(f); display(Video(f, embed=True, width=270))

In [ ]:
#@title 5. Télécharger tous les clips (zip)
!cd outputs && zip -q -r ../clips_dj.zip highlight_rank_*_ready.mp4 thumbnail_rank_*.jpg dj_energy.png metadata_preview.json
from google.colab import files
files.download("clips_dj.zip")

## Ajuster un clip à la main (optionnel)

Le fichier `outputs/gemini_response.json` contient la sélection (`start_time`, `end_time`, `hook_start_time` en secondes).
Modifie-le (panneau Fichiers à gauche), puis relance avec la cellule ci-dessous : seul le rendu est refait.

In [ ]:
#@title 6. Re-rendre après modification manuelle
!{cmd} --load-gemini-json